# 01. TIỀN XỬ LÝ DỮ LIỆU USD/VND

Notebook này giữ nguyên quy trình của bộ gốc nhưng được viết lại cho file lịch sử **USD/VND**.

Dữ liệu gốc có các cột tiếng Việt:
- `Ngày`
- `Lần cuối`
- `Mở`
- `Cao`
- `Thấp`
- `KL`
- `% Thay đổi`

Sau khi đọc dữ liệu, các cột cần dùng được chuẩn hóa về `Date`, `Close`, `Open`, `High`, `Low`.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==============================
# 1. THIẾT LẬP ĐƯỜNG DẪN
# ==============================

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "code":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

file_path = RAW_DIR / "USD_VND_history.csv"

if not file_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu gốc: {file_path}\n"
        "Hãy đặt file USD_VND_history.csv trong thư mục data/raw/."
    )

df = pd.read_csv(file_path)

print("5 dòng đầu:")
print(df.head())

print("\nKích thước dữ liệu:")
print(df.shape)

print("\nTên các cột:")
print(df.columns.tolist())

print("\nKiểu dữ liệu ban đầu:")
print(df.dtypes)


5 dòng đầu:
         Ngày  Lần cuối        Mở       Cao      Thấp   KL % Thay đổi
0  31/08/2026  26,072.5  26,072.5  26,072.5  26,072.5  NaN      0.00%
1  28/08/2026  26,072.5  26,130.5  26,130.5  26,072.5  NaN     -0.02%
2  27/08/2026  26,078.0  26,102.0  26,105.0  26,064.5  NaN     -0.10%
3  26/08/2026  26,105.0  26,109.5  26,142.5  26,072.0  NaN     -0.03%
4  25/08/2026  26,112.0  26,179.5  26,188.5  26,098.0  NaN     -0.23%

Kích thước dữ liệu:
(2784, 7)

Tên các cột:
['Ngày', 'Lần cuối', 'Mở', 'Cao', 'Thấp', 'KL', '% Thay đổi']

Kiểu dữ liệu ban đầu:
Ngày          object
Lần cuối      object
Mở            object
Cao           object
Thấp          object
KL            object
% Thay đổi    object
dtype: object


In [2]:
# ==============================
# 2. ĐỔI TÊN CỘT VÀ CHUẨN HÓA KIỂU DỮ LIỆU
# ==============================

rename_map = {
    "Ngày": "Date",
    "Lần cuối": "Close",
    "Mở": "Open",
    "Cao": "High",
    "Thấp": "Low",
    "KL": "Volume",
    "% Thay đổi": "Change_pct"
}

df = df.rename(columns=rename_map)

required_cols = ["Date", "Close", "Open", "High", "Low"]
missing_required = [c for c in required_cols if c not in df.columns]

if missing_required:
    raise ValueError(
        f"Thiếu các cột bắt buộc sau khi đổi tên: {missing_required}"
    )

# Ngày trong file gốc có dạng dd/mm/YYYY
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d/%m/%Y",
    errors="coerce"
)

# Các cột giá có dạng chuỗi như '26,072.5'
price_cols = ["Open", "High", "Low", "Close"]

for col in price_cols:
    df[col] = pd.to_numeric(
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

# Chuẩn hóa % thay đổi nếu cột có tồn tại
if "Change_pct" in df.columns:
    df["Change_pct"] = pd.to_numeric(
        df["Change_pct"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

print(df.dtypes)
print("Ngày đầu:", df["Date"].min())
print("Ngày cuối:", df["Date"].max())


Date          datetime64[ns]
Close                float64
Open                 float64
High                 float64
Low                  float64
Volume                object
Change_pct           float64
dtype: object
Ngày đầu: 2016-01-04 00:00:00
Ngày cuối: 2026-08-31 00:00:00


In [3]:
# Sắp ngày cũ -> mới
df = df.sort_values("Date").reset_index(drop=True)

print(df.head())
print(df.tail())


        Date    Close     Open     High      Low Volume  Change_pct
0 2016-01-04  22424.0  22490.0  22524.5  22438.0    NaN       -0.25
1 2016-01-05  22458.0  22505.0  22507.0  22445.0    NaN        0.15
2 2016-01-06  22465.0  22472.5  22507.5  22465.0    NaN        0.03
3 2016-01-07  22428.0  22495.0  22510.0  22445.0    NaN       -0.16
4 2016-01-08  22426.0  22475.5  22489.0  22450.5    NaN       -0.01
           Date    Close     Open     High      Low Volume  Change_pct
2779 2026-08-25  26112.0  26179.5  26188.5  26098.0    NaN       -0.23
2780 2026-08-26  26105.0  26109.5  26142.5  26072.0    NaN       -0.03
2781 2026-08-27  26078.0  26102.0  26105.0  26064.5    NaN       -0.10
2782 2026-08-28  26072.5  26130.5  26130.5  26072.5    NaN       -0.02
2783 2026-08-31  26072.5  26072.5  26072.5  26072.5    NaN        0.00


In [4]:
# ==============================
# 3. KIỂM TRA MISSING
# ==============================

print("\nSố missing của từng biến:")
print(df.isnull().sum())



Số missing của từng biến:
Date             0
Close            0
Open             0
High             0
Low              0
Volume        1777
Change_pct       0
dtype: int64


In [5]:
# ==============================
# 4. KIỂM TRA NGÀY BỊ TRÙNG
# ==============================

duplicates = df[df["Date"].duplicated(keep=False)]

print("\nSố ngày bị trùng:")
print(len(duplicates))

if len(duplicates) > 0:
    print(duplicates)



Số ngày bị trùng:
0


In [6]:
# ==============================
# 5. KIỂM TRA GIÁ KHÔNG HỢP LỆ
# ==============================

price_cols = ["Open", "High", "Low", "Close"]

invalid_price = df[
    (df[price_cols] <= 0).any(axis=1)
]

print("\nCác dòng có giá <= 0:")
print(invalid_price)



Các dòng có giá <= 0:
Empty DataFrame
Columns: [Date, Close, Open, High, Low, Volume, Change_pct]
Index: []


In [7]:
# ==============================
# 6. KIỂM TRA LOGIC OHLC
# ==============================

invalid_ohlc = df[
    (df["High"] < df["Open"]) |
    (df["High"] < df["Close"]) |
    (df["High"] < df["Low"]) |
    (df["Low"] > df["Open"]) |
    (df["Low"] > df["Close"])
]

print("\nSố dòng không thỏa logic OHLC:")
print(len(invalid_ohlc))

if len(invalid_ohlc) > 0:
    print(invalid_ohlc.head(20))

print(
    "\nLưu ý: bộ phân tích chính dùng Close. "
    "Các dòng OHLC bị gắn cờ để kiểm tra nguồn, không tự động xóa."
)



Số dòng không thỏa logic OHLC:
569
         Date    Close     Open     High      Low Volume  Change_pct
0  2016-01-04  22424.0  22490.0  22524.5  22438.0    NaN       -0.25
3  2016-01-07  22428.0  22495.0  22510.0  22445.0    NaN       -0.16
4  2016-01-08  22426.0  22475.5  22489.0  22450.5    NaN       -0.01
5  2016-01-11  22418.0  22475.0  22505.0  22419.5    NaN       -0.04
10 2016-01-18  22365.0  22430.5  22467.5  22376.5    NaN       -0.22
11 2016-01-19  22392.0  22422.5  22477.0  22401.0    NaN        0.12
12 2016-01-20  22396.0  22425.0  22477.0  22408.0    NaN        0.02
13 2016-01-21  22336.0  22423.0  22475.0  22376.5    NaN       -0.27
15 2016-01-25  22314.0  22430.5  22476.0  22348.5    NaN       -0.35
17 2016-01-27  22268.0  22305.0  22330.0  22284.0    NaN       -0.16
18 2016-01-28  22171.0  22257.5  22300.0  22213.0    NaN       -0.44
19 2016-01-29  22198.0  22233.5  22257.5  22214.0    NaN        0.12
22 2016-02-03  22219.0  22294.0  22314.5  22257.0    NaN       -0.2

In [8]:
# ==============================
# 7. KIỂM TRA KHOẢNG CÁCH NGÀY
# ==============================

df["day_gap"] = df["Date"].diff().dt.days

print(df["day_gap"].value_counts().sort_index())


day_gap
1.0    2227
2.0       3
3.0     553
Name: count, dtype: int64


In [9]:
# Biến giá chính dùng cho mô hình
df["price"] = df["Close"]


In [10]:
# ==============================
# 8. TẠO LOG PRICE
# ==============================

df["log_price"] = np.log(df["Close"])


In [11]:
# ==============================
# 9. TẠO RETURN VÀ LOG-RETURN
# ==============================

df["simple_return"] = df["Close"].pct_change() * 100

df["log_return"] = (
    np.log(df["Close"] / df["Close"].shift(1)) * 100
)

print(df.head())


        Date    Close     Open  ...  log_price  simple_return log_return
0 2016-01-04  22424.0  22490.0  ...  10.017887            NaN        NaN
1 2016-01-05  22458.0  22505.0  ...  10.019402       0.151623   0.151508
2 2016-01-06  22465.0  22472.5  ...  10.019714       0.031169   0.031164
3 2016-01-07  22428.0  22495.0  ...  10.018065      -0.164701  -0.164836
4 2016-01-08  22426.0  22475.5  ...  10.017976      -0.008917  -0.008918

[5 rows x 12 columns]


In [12]:
# Biến hỗ trợ phân tích volatility
df["squared_return"] = df["log_return"] ** 2
df["abs_return"] = df["log_return"].abs()


In [13]:
# ==============================
# 10. GẮN CỜ OUTLIER LOG-RETURN BẰNG IQR
# ==============================

returns = df["log_return"].dropna()

Q1 = returns.quantile(0.25)
Q3 = returns.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df["outlier_flag"] = (
    (df["log_return"] < lower_bound) |
    (df["log_return"] > upper_bound)
)

print("Ngưỡng dưới:", lower_bound)
print("Ngưỡng trên :", upper_bound)

print(
    df[df["outlier_flag"]][
        ["Date", "Close", "log_return"]
    ]
)


Ngưỡng dưới: -0.1553905064815899
Ngưỡng trên : 0.15774398389692523
           Date    Close  log_return
3    2016-01-07  22428.0   -0.164836
7    2016-01-13  22393.0   -0.156177
8    2016-01-14  22436.0    0.191840
10   2016-01-18  22365.0   -0.223314
13   2016-01-21  22336.0   -0.268264
...         ...      ...         ...
2771 2026-08-14  26148.5    0.281483
2772 2026-08-17  26205.0    0.215840
2775 2026-08-20  26102.0   -0.300292
2778 2026-08-24  26173.5    0.187388
2779 2026-08-25  26112.0   -0.235247

[381 rows x 3 columns]


In [14]:
# ==============================
# 11. THỐNG KÊ NHANH
# ==============================

print("\nThống kê tỷ giá USD/VND:")
print(df["Close"].describe())

print("\nThống kê log-return:")
print(df["log_return"].describe())



Thống kê tỷ giá USD/VND:
count     2784.00000
mean     23743.63380
std       1258.72154
min      22171.00000
25%      22785.75000
50%      23208.00000
75%      24560.00000
max      26432.50000
Name: Close, dtype: float64

Thống kê log-return:
count    2783.000000
mean        0.005417
std         0.136566
min        -1.531170
25%        -0.037965
50%         0.000000
75%         0.040319
max         1.175292
Name: log_return, dtype: float64


In [15]:
# ==============================
# 12. TẠO DATA GIÁ SẠCH
# ==============================

usd_vnd_price = df[
    [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "log_price"
    ]
].copy()

print(usd_vnd_price.head())
print(usd_vnd_price.isnull().sum())


        Date     Open     High      Low    Close  log_price
0 2016-01-04  22490.0  22524.5  22438.0  22424.0  10.017887
1 2016-01-05  22505.0  22507.0  22445.0  22458.0  10.019402
2 2016-01-06  22472.5  22507.5  22465.0  22465.0  10.019714
3 2016-01-07  22495.0  22510.0  22445.0  22428.0  10.018065
4 2016-01-08  22475.5  22489.0  22450.5  22426.0  10.017976
Date         0
Open         0
High         0
Low          0
Close        0
log_price    0
dtype: int64


In [16]:
# ==============================
# 13. TẠO DATA RETURN SẠCH
# ==============================

usd_vnd_return = df[
    [
        "Date",
        "Close",
        "log_price",
        "simple_return",
        "log_return",
        "squared_return",
        "abs_return"
    ]
].copy()

usd_vnd_return = usd_vnd_return.dropna(
    subset=["log_return"]
).reset_index(drop=True)

print(usd_vnd_return.head())
print(usd_vnd_return.isnull().sum())


        Date    Close  log_price  ...  log_return  squared_return  abs_return
0 2016-01-05  22458.0  10.019402  ...    0.151508        0.022955    0.151508
1 2016-01-06  22465.0  10.019714  ...    0.031164        0.000971    0.031164
2 2016-01-07  22428.0  10.018065  ...   -0.164836        0.027171    0.164836
3 2016-01-08  22426.0  10.017976  ...   -0.008918        0.000080    0.008918
4 2016-01-11  22418.0  10.017619  ...   -0.035679        0.001273    0.035679

[5 rows x 7 columns]
Date              0
Close             0
log_price         0
simple_return     0
log_return        0
squared_return    0
abs_return        0
dtype: int64


In [17]:
# ==============================
# 14. LƯU DỮ LIỆU ĐÃ TIỀN XỬ LÝ
# ==============================

usd_vnd_price.to_csv(
    PROCESSED_DIR / "usd_vnd_price_clean.csv",
    index=False
)

usd_vnd_return.to_csv(
    PROCESSED_DIR / "usd_vnd_return_clean.csv",
    index=False
)

print("\nĐã lưu dữ liệu đã tiền xử lý:")
print(PROCESSED_DIR / "usd_vnd_price_clean.csv")
print(PROCESSED_DIR / "usd_vnd_return_clean.csv")



Đã lưu dữ liệu đã tiền xử lý:
/mnt/data/USD_VND_notebooks/data/processed/usd_vnd_price_clean.csv
/mnt/data/USD_VND_notebooks/data/processed/usd_vnd_return_clean.csv
